# Stage 1 — FraudZen forensic preprocessing

This notebook parses the selected `all_naive_12%_200sims` scenario, audits data quality, removes cross-file duplicate events, and attaches ground-truth labels. It does not create a fraud score or train a model.

In [1]:
from pathlib import Path
from datetime import datetime
import csv
from collections import Counter

SCENARIO = Path(r'../Fraudzen-dataset/Fraudzen-dataset/FraudZen_Cdrs/all_naive/all_naive_12%_200sims')
OUTPUT = Path('../outputs/processed/all_naive_12pct_200sims')
OUTPUT.mkdir(parents=True, exist_ok=True)

RAW_COLUMNS = ['timestamp', 'cdr_type', 'src_number', 'src_imei',
               'initial_src_cell_id', 'current_src_cell_id', 'dst_number',
               'dst_imei', 'initial_dst_cell_id', 'current_dst_cell_id',
               'duration', 'size']
OUTPUT_COLUMNS = RAW_COLUMNS + ['dataset', 'record_id', 'event_key',
                  'src_known_fraud', 'dst_known_fraud', 'involves_known_fraud']

fraud_users = {x.strip() for x in (SCENARIO / 'fraudulentUsers.txt').read_text(encoding='utf-8').splitlines() if x.strip()}
print('Ground-truth users:', len(fraud_users))

Ground-truth users: 200


In [2]:
def parse_file(path, dataset):
    parsed, malformed, variants, timestamp_failures = [], [], [], []
    with path.open(encoding='utf-8', errors='replace') as handle:
        header = handle.readline().strip().split()
        for line_number, raw in enumerate(handle, start=2):
            if not raw.strip():
                continue
            fields = raw.strip().split()
            if len(fields) not in (12, 13):
                malformed.append({'source_line': line_number, 'field_count': len(fields), 'raw_line': raw.rstrip()})
                continue
            row = {
                'timestamp': f'{fields[0]} {fields[1]}', 'cdr_type': fields[2],
                'src_number': fields[3].strip(), 'src_imei': fields[4].strip(),
                'initial_src_cell_id': fields[5], 'current_src_cell_id': fields[6],
                'dst_number': fields[7].strip(), 'dst_imei': fields[8].strip(),
                'initial_dst_cell_id': fields[9], 'current_dst_cell_id': fields[10],
                'duration': fields[11], 'size': fields[12] if len(fields) == 13 else '',
                'dataset': dataset, 'source_line': line_number}
            if len(fields) == 12:
                variants.append({'source_line': line_number, 'field_count': len(fields), 'raw_line': raw.rstrip()})
            try:
                row['timestamp'] = datetime.strptime(row['timestamp'], '%d-%m-%Y %H:%M:%S').isoformat(sep=' ')
            except ValueError:
                timestamp_failures.append({'source_line': line_number, 'raw_timestamp': row['timestamp'], 'raw_line': raw.rstrip()})
                continue
            parsed.append(row)
    return parsed, malformed, variants, timestamp_failures, header

def write_csv(path, rows, fieldnames):
    with path.open('w', newline='', encoding='utf-8') as handle:
        writer = csv.DictWriter(handle, fieldnames=fieldnames, extrasaction='ignore')
        writer.writeheader()
        writer.writerows(rows)

In [3]:
all_rows, audit = [], []
for filename, dataset in [('International_CDRTrace.txt', 'international'), ('Op_1_CDRTrace.txt', 'local')]:
    parsed, malformed, variants, failures, header = parse_file(SCENARIO / filename, dataset)
    audit.append({'dataset': dataset, 'filename': filename, 'header': ' | '.join(header),
                  'parsed': len(parsed), 'malformed': len(malformed),
                  'schema_variants': len(variants), 'timestamp_failures': len(failures)})
    write_csv(OUTPUT / f'{dataset}_malformed_rows.csv', malformed, ['source_line', 'field_count', 'raw_line'])
    write_csv(OUTPUT / f'{dataset}_schema_variant_rows.csv', variants, ['source_line', 'field_count', 'raw_line'])
    write_csv(OUTPUT / f'{dataset}_timestamp_failures.csv', failures, ['source_line', 'raw_timestamp', 'raw_line'])
    all_rows.extend(parsed)

print('Parsed events:', len(all_rows))
audit

Parsed events: 88416


[{'dataset': 'international',
  'filename': 'International_CDRTrace.txt',
  'header': 'timestamp | CDR_type | src_number | src_imei | initial_src_cellId | current_src_cellId | dst_number | dst_imei | initial_dst_cellId | current_dst_cellId | duration | size',
  'parsed': 1543,
  'malformed': 0,
  'schema_variants': 1543,
  'timestamp_failures': 0},
 {'dataset': 'local',
  'filename': 'Op_1_CDRTrace.txt',
  'header': 'timestamp | CDR_type | src_number | src_imei | initial_src_cellId | current_src_cellId | dst_number | dst_imei | initial_dst_cellId | current_dst_cellId | duration | size',
  'parsed': 86873,
  'malformed': 0,
  'schema_variants': 0,
  'timestamp_failures': 0}]

In [4]:
international_keys = {(r['timestamp'], r['src_number'], r['dst_number'], r['duration']) for r in all_rows if r['dataset'] == 'international'}
canonical, duplicates = [], []
seen = set()
for row in all_rows:
    key = (row['timestamp'], row['src_number'], row['dst_number'], row['duration'])
    if row['dataset'] == 'local' and key in international_keys:
        duplicates.append({**row, 'event_key': '|'.join(key), 'duplicate_reason': 'cross_file_international_copy'})
        continue
    if key in seen:
        duplicates.append({**row, 'event_key': '|'.join(key), 'duplicate_reason': 'repeated_canonical_event'})
        continue
    seen.add(key)
    row['event_key'] = '|'.join(key)
    row['src_known_fraud'] = row['src_number'] in fraud_users
    row['dst_known_fraud'] = row['dst_number'] in fraud_users
    row['involves_known_fraud'] = row['src_known_fraud'] or row['dst_known_fraud']
    canonical.append(row)

for record_id, row in enumerate(canonical):
    row['record_id'] = record_id

write_csv(OUTPUT / 'canonical_cdr.csv', canonical, OUTPUT_COLUMNS)
write_csv(OUTPUT / 'excluded_duplicate_events.csv', duplicates, OUTPUT_COLUMNS + ['duplicate_reason'])
write_csv(OUTPUT / 'fraudulent_users.csv', [{'src_number': x, 'known_fraud': 1} for x in sorted(fraud_users)], ['src_number', 'known_fraud'])

with (OUTPUT / 'preprocessing_summary.csv').open('w', newline='', encoding='utf-8') as handle:
    writer = csv.DictWriter(handle, fieldnames=audit[0].keys())
    writer.writeheader(); writer.writerows(audit)

print('Canonical events:', len(canonical))
print('Excluded duplicates:', len(duplicates))
print('Source-fraud events:', sum(r['src_known_fraud'] for r in canonical))
print('Destination-fraud events:', sum(r['dst_known_fraud'] for r in canonical))

Canonical events: 86873
Excluded duplicates: 1543
Source-fraud events: 662
Destination-fraud events: 0


## Expected result

For the selected scenario, the current validated result is 88,416 parsed events, 86,873 canonical events, and 1,543 excluded cross-file duplicates.